In [1]:
import jax
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
# ````````````````````````````````````````````````````````````````````
# This section combine two folders into one with shuffle 
# ````````````````````````````````````````````````````````````````````
import os
import shutil
import random  # Step 1: Import random
from pathlib import Path

def combine_and_relabel_random(source_dirs, target_dir):
    target_path = Path(target_dir)
    target_path.mkdir(parents=True, exist_ok=True)
    
    print(f"Target Directory: {target_path.absolute()}")
    
    # Step 2: Collect ALL session paths into one list first
    all_session_paths = []
    for source in source_dirs:
        source_path = Path(source)
        if not source_path.exists():
            continue
        # Find all session subdirectories
        sessions = [d for d in source_path.iterdir() if d.is_dir()]
        all_session_paths.extend(sessions)

    # Step 3: Randomize the list
    print(f"Shuffling {len(all_session_paths)} sessions...")
    random.shuffle(all_session_paths)
    
    # Step 4: Iterate through the shuffled list and label them sequentially
    for idx, session_folder in enumerate(all_session_paths, start=1):
        new_name = f"session_{idx:03d}"
        dest_path = target_path / new_name
        
        print(f"  [{idx:03d}] Copying: {session_folder.parent.name}/{session_folder.name} -> {new_name}")
        shutil.copytree(session_folder, dest_path)

    print(f"\nDone! Randomly combined {len(all_session_paths)} sessions into {target_dir}")

# --- Configuration remains the same ---
folders_to_combine = [
    "/home/lenovo/Downloads/2026-04-30-full",
    "/home/lenovo/Downloads/combined_dataset_pick_toy_right_arm3"
]
output_folder = "/home/lenovo/Downloads/combined_dataset_pick_toy_right_arm4"

combine_and_relabel_random(folders_to_combine, output_folder)

Target Directory: /home/lenovo/Downloads/combined_dataset_pick_toy_right_arm4
Shuffling 181 sessions...
  [001] Copying: combined_dataset_pick_toy_right_arm3/session_104 -> session_001
  [002] Copying: 2026-04-30-full/session_010 -> session_002
  [003] Copying: combined_dataset_pick_toy_right_arm3/session_008 -> session_003
  [004] Copying: 2026-04-30-full/session_021 -> session_004
  [005] Copying: 2026-04-30-full/session_024 -> session_005
  [006] Copying: combined_dataset_pick_toy_right_arm3/session_092 -> session_006
  [007] Copying: combined_dataset_pick_toy_right_arm3/session_064 -> session_007
  [008] Copying: combined_dataset_pick_toy_right_arm3/session_090 -> session_008
  [009] Copying: combined_dataset_pick_toy_right_arm3/session_047 -> session_009
  [010] Copying: combined_dataset_pick_toy_right_arm3/session_134 -> session_010
  [011] Copying: combined_dataset_pick_toy_right_arm3/session_117 -> session_011
  [012] Copying: combined_dataset_pick_toy_right_arm3/session_005 ->

In [ ]:
# `````````````````````````````````````````````````````````````````````````````
# for single session test
# `````````````````````````````````````````````````````````````````````````````

import os
import shutil
from pathlib import Path

def flatten_and_rename_video_folders(base_directory):
    base_path = Path(base_directory)
    
    if not base_path.exists():
        print(f"Error: The directory '{base_directory}' does not exist.")
        return

    moved_count = 0

    # Iterate through all items in the base videos folder
    for subfolder in base_path.iterdir():
        # Only process actual directories
        if subfolder.is_dir():
            folder_name = subfolder.name
            print(f"Processing folder: {folder_name}")
            
            # Find all files directly inside or recursively within this subfolder
            for file_path in subfolder.rglob("*"):
                if file_path.is_file():
                    # Prefix the filename with the subfolder's name
                    new_filename = f"{folder_name}"
                    target_path = base_path / new_filename
                    
                    # Safety check for duplicate names (just in case)
                    if target_path.exists():
                        base_name = Path(new_filename).stem
                        extension = file_path.suffix
                        counter = 1
                        while target_path.exists():
                            target_path = base_path / f"{base_name}{extension}"
                            counter += 1
                    
                    # Move and rename the file up to the main videos folder
                    shutil.move(str(file_path), str(target_path))
                    moved_count += 1
            
            # Clean up the empty subfolder
            try:
                subfolder.rmdir()
                print(f"Successfully cleaned up empty folder: {folder_name}")
            except OSError:
                print(f"Note: Could not remove {folder_name}. It might still contain nested structures.")

    print("\n" + "="*50)
    print(f"Done! Consolidated {moved_count} files with folder prefixes into '{base_directory}'.")
    print("="*50)

# --- CONFIGURATION ---
VIDEOS_DIR = "./videos" 

if __name__ == "__main__":
    flatten_and_rename_video_folders("/home/lenovo/Downloads/pick_cube_task/episode_000018/videos")

Processing folder: rgbd_head_depth
Successfully cleaned up empty folder: rgbd_head_depth
Processing folder: hand_right
Successfully cleaned up empty folder: hand_right
Processing folder: hand_left
Successfully cleaned up empty folder: hand_left
Processing folder: rgbd_head_color
Successfully cleaned up empty folder: rgbd_head_color

Done! Consolidated 4 files with folder prefixes into '/home/lenovo/Downloads/pick_cube_task/episode_000018/videos'.


In [ ]:
# `````````````````````````````````````````````````````````````````````````````
# This section moves all videos from subfolders into 'data_dir/videos'. 
# Rename each video after its original folder, then delete the empty folders. 
# `````````````````````````````````````````````````````````````````````````````

import os
import shutil
from pathlib import Path

def flatten_and_rename_single_video_dir(videos_path: Path):
    """Processes a single 'videos' folder, naming files based on their immediate subfolder."""
    print(f"\n--- Processing Video Directory: {videos_path} ---")
    moved_count = 0

    # Iterate through subfolders inside this specific videos directory
    for subfolder in videos_path.iterdir():
        if subfolder.is_dir():
            folder_name = subfolder.name
            
            # Find all files recursively inside this subfolder
            for file_path in subfolder.rglob("*"):
                if file_path.is_file():
                    # FIX 1: Keep the original extension (e.g., 'subfolder_clip.mp4')
                    extension = file_path.suffix
                    base_stem = file_path.stem
                    new_filename = f"{folder_name}{extension}"
                    target_path = videos_path / new_filename
                    
                    # FIX 2: Fixed the infinite loop logic for duplicates
                    if target_path.exists():
                        counter = 1
                        while target_path.exists():
                            target_path = videos_path / f"{folder_name}{extension}"
                            counter += 1
                    
                    # Move and rename the file up to its parent videos folder
                    shutil.move(str(file_path), str(target_path))
                    moved_count += 1
            
            # Clean up the empty subfolder
            try:
                subfolder.rmdir()
                print(f"  Cleaned up empty subfolder: {folder_name}")
            except OSError:
                print(f"  Note: Could not remove {folder_name}. It may not be empty.")

    print(f"Result: Moved {moved_count} files inside {videos_path.parent.name}/videos")

def batch_process_all_episodes(root_directory):
    root_path = Path(root_directory)
    
    if not root_path.exists():
        print(f"Error: Target path '{root_directory}' does not exist.")
        return

    # Find every folder named 'videos' anywhere inside the root directory structure
    # This automatically matches episode_000000/videos, episode_000001/videos, etc.
    videos_directories = list(root_path.rglob("videos"))
    
    if not videos_directories:
        print("No 'videos' folders found in the specified path structure.")
        return

    print(f"Found {len(videos_directories)} 'videos' directories to process.")
    
    for v_dir in videos_directories:
        if v_dir.is_dir():
            flatten_and_rename_single_video_dir(v_dir)

    print("\n" + "="*60)
    print("Batch processing complete for all session datasets!")
    print("="*60)

# --- CONFIGURATION ---
# Point this to the root tasks directory containing all your episode_xxxxxx folders
DATASET_ROOT = "/home/lenovo/Downloads/pick_cube_task (Copy)"

if __name__ == "__main__":
    batch_process_all_episodes("/home/lenovo/Downloads/pick_cube_task (Copy)")

Found 50 'videos' directories to process.

--- Processing Video Directory: /home/lenovo/Downloads/pick_cube_task (Copy)/episode_000023/videos ---
  Cleaned up empty subfolder: rgbd_head_depth
  Cleaned up empty subfolder: hand_right
  Cleaned up empty subfolder: hand_left
  Cleaned up empty subfolder: rgbd_head_color
Result: Moved 4 files inside episode_000023/videos

--- Processing Video Directory: /home/lenovo/Downloads/pick_cube_task (Copy)/episode_000000/videos ---
Result: Moved 0 files inside episode_000000/videos

--- Processing Video Directory: /home/lenovo/Downloads/pick_cube_task (Copy)/episode_000035/videos ---
  Cleaned up empty subfolder: rgbd_head_depth
  Cleaned up empty subfolder: hand_right
  Cleaned up empty subfolder: hand_left
  Cleaned up empty subfolder: rgbd_head_color
Result: Moved 4 files inside episode_000035/videos

--- Processing Video Directory: /home/lenovo/Downloads/pick_cube_task (Copy)/episode_000006/videos ---
Result: Moved 0 files inside episode_000006

In [ ]:
# ```````````````````````````````````````````````````````````````````````````````````
# This section moves all "manifest.jsonl" file from data_dir into 'data_dir/joints'. 
# ```````````````````````````````````````````````````````````````````````````````````

import shutil
from pathlib import Path

def fix_and_clean_manifests(root_directory):
    root_path = Path(root_directory)
    
    if not root_path.exists():
        print(f"Error: Path '{root_directory}' does not exist.")
        return

    print("Starting clean-up and proper organization...")
    fixed_count = 0

    # Step 1: Scan every episode directory directly (non-recursive loop to avoid inception)
    for episode_dir in root_path.iterdir():
        if episode_dir.is_dir() and episode_dir.name.startswith("episode_"):
            
            # Find ANY manifest.jsonl hidden inside this episode structure
            all_manifests = list(episode_dir.rglob("manifest.jsonl"))
            
            if not all_manifests:
                continue
                
            # Take the first one found (the real data)
            actual_manifest = all_manifests[0]
            
            # Define the exact clean path we want: episode_xxxxxx/joints/manifest.jsonl
            correct_joints_dir = episode_dir / "joints"
            final_target_file = correct_joints_dir / "manifest.jsonl"
            
            # Temporarily move the file to a safe mid-way location out of the way
            temp_safe_path = episode_dir / "temp_manifest.jsonl"
            shutil.move(str(actual_manifest), str(temp_safe_path))
            
            # Step 2: Nuke all existing nested 'joints' folders inside this episode
            for item in list(episode_dir.iterdir()):
                if item.is_dir() and item.name == "joints":
                    try:
                        shutil.rmtree(str(item))
                    except Exception as e:
                        print(f"Could not delete old folder {item}: {e}")
            
            # Step 3: Recreate the single, clean 'joints' folder
            correct_joints_dir.mkdir(exist_ok=True)
            
            # Step 4: Move the manifest into its true home
            shutil.move(str(temp_safe_path), str(final_target_file))
            print(f"Fixed: {episode_dir.name}/joints/manifest.jsonl")
            fixed_count += 1

    print("\n" + "="*60)
    print(f"All done! Successfully rescued and cleaned {fixed_count} episodes.")
    print("="*60)

# --- CONFIGURATION ---
DATASET_ROOT = "/home/lenovo/Downloads/pick_cube_task"

if __name__ == "__main__":
    fix_and_clean_manifests(DATASET_ROOT)

Starting clean-up and proper organization...
Fixed: episode_000023/joints/manifest.jsonl
Fixed: episode_000000/joints/manifest.jsonl
Fixed: episode_000035/joints/manifest.jsonl
Fixed: episode_000006/joints/manifest.jsonl
Fixed: episode_000020/joints/manifest.jsonl
Fixed: episode_000011/joints/manifest.jsonl
Fixed: episode_000027/joints/manifest.jsonl
Fixed: episode_000032/joints/manifest.jsonl
Fixed: episode_000024/joints/manifest.jsonl
Fixed: episode_000002/joints/manifest.jsonl
Fixed: episode_000039/joints/manifest.jsonl
Fixed: episode_000001/joints/manifest.jsonl
Fixed: episode_000012/joints/manifest.jsonl
Fixed: episode_000029/joints/manifest.jsonl
Fixed: episode_000028/joints/manifest.jsonl
Fixed: episode_000016/joints/manifest.jsonl
Fixed: episode_000017/joints/manifest.jsonl
Fixed: episode_000045/joints/manifest.jsonl
Fixed: episode_000022/joints/manifest.jsonl
Fixed: episode_000003/joints/manifest.jsonl
Fixed: episode_000004/joints/manifest.jsonl
Fixed: episode_000040/joints/ma

In [ ]:
# `````````````````````````````````````````````````````````````````````````````
# This section removes all "manifest.jsonl" header. 
# `````````````````````````````````````````````````````````````````````````````

import json
from pathlib import Path

def remove_headers_from_manifests(root_directory):
    root_path = Path(root_directory)
    
    if not root_path.exists():
        print(f"Error: Path '{root_directory}' does not exist.")
        return

    modified_files_count = 0

    # Recursively find all manifest.jsonl files
    for manifest_path in root_path.rglob("manifest.jsonl"):
        if manifest_path.is_file():
            
            # Read all lines from the current file
            with open(manifest_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            
            if not lines:
                continue
                
            try:
                # Parse the first line to check if it's the header
                first_line_json = json.loads(lines[0].strip())
                
                if first_line_json.get("_type") == "session_header":
                    # Keep everything EXCEPT the first line
                    cleaned_lines = lines[1:]
                    
                    # Overwrite the file with the cleaned lines
                    with open(manifest_path, 'w', encoding='utf-8') as f:
                        f.writelines(cleaned_lines)
                        
                    print(f"Successfully removed header from: {manifest_path.relative_to(root_path)}")
                    modified_files_count += 1
                else:
                    print(f"Skipped (No header found): {manifest_path.relative_to(root_path)}")
                    
            except json.JSONDecodeError:
                print(f"Warning: Could not parse first line of {manifest_path.name}. Skipping.")

    print("\n" + "="*60)
    print(f"Done! Cleaned the session header out of {modified_files_count} files.")
    print("="*60)

# --- CONFIGURATION ---
DATASET_ROOT = "/home/lenovo/Downloads/pick_cube_task"

if __name__ == "__main__":
    remove_headers_from_manifests(DATASET_ROOT)

Successfully removed header from: episode_000023/joints/manifest.jsonl
Skipped (No header found): episode_000000/joints/manifest.jsonl
Successfully removed header from: episode_000035/joints/manifest.jsonl
Successfully removed header from: episode_000006/joints/manifest.jsonl
Successfully removed header from: episode_000020/joints/manifest.jsonl
Successfully removed header from: episode_000011/joints/manifest.jsonl
Successfully removed header from: episode_000027/joints/manifest.jsonl
Successfully removed header from: episode_000032/joints/manifest.jsonl
Successfully removed header from: episode_000024/joints/manifest.jsonl
Successfully removed header from: episode_000002/joints/manifest.jsonl
Successfully removed header from: episode_000039/joints/manifest.jsonl
Successfully removed header from: episode_000001/joints/manifest.jsonl
Successfully removed header from: episode_000012/joints/manifest.jsonl
Successfully removed header from: episode_000029/joints/manifest.jsonl
Successfully 